# Braindecode SOTA 实验 (独立于原始基线)

基于 Braindecode 模型库的 EEG 分类实验。

- SLEEP  → **USleep** (专为睡眠分期设计)
- BCIC2A → **ShallowFBCSPNet** (运动想象黄金标准)
- MDD / SEED / CHINESE → **Deep4Net** (通用深度 CNN)

**结果写入独立目录** `_results_braindecode/`，与原 `_results/` 完全隔离。

---
## 0. Colab 环境准备

In [ ]:
# Kaggle: skip Google Drive/Colab-only setup
import os
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = (not IS_KAGGLE) and os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")
else:
    print("Kaggle/non-Colab environment, skip Drive mount.")


---
## 1. 实验配置

In [ ]:
# ================================================================
# Formal experiment configuration
# ================================================================

# Chinese is excluded from leaderboard analysis per instructor guidance.
DATASETS = ["BCIC2A", "MDD", "SEED", "SLEEP"]
EXCLUDED_DATASETS = {"CHINESE"}

# Formal runs are separated from smoke runs in names and summary files.
RUN_TAG = "formal"

MODEL_MAP = {
    "SLEEP":   "USleep",
    "BCIC2A":  "ShallowFBCSPNet",
    "MDD":     "Deep4Net",
    "SEED":    "Deep4Net",
}

# Lightweight tuned formal training, with early stopping and cosine scheduling.
EPOCHS = 80
BATCH_SIZE = 32
LR = 1e-3
PATIENCE = 15
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0
SEED = 42

# Dataset-level training overrides.
DATASET_TRAINING_OVERRIDES = {
    "BCIC2A": {"lr": 3e-4, "epochs": 120, "patience": 25},
    "MDD":    {"lr": 1e-3, "epochs": 80,  "patience": 15},
    "SEED":   {"lr": 5e-4, "epochs": 100, "patience": 20,
               "weight_decay": 5e-4, "label_smoothing": 0.02},
    "SLEEP":  {"lr": 5e-4, "epochs": 100, "patience": 20,
               "weight_decay": 1e-4, "label_smoothing": 0.02},
}

# Keep False for fresh formal training. If changed to True, only formal runs
# whose directory names contain RUN_TAG can block a rerun; smoke1ep runs cannot.
SKIP_EXISTING = False

DATA_ROOT_OVERRIDE = None
USE_LOCAL_CACHE = True
SYNC_TO_DRIVE = False

# Kaggle quota guard: preflight first, and never train on CPU by accident.
PREFLIGHT_ONLY = True
REQUIRE_GPU = True

print("=== Braindecode formal training config ===")
print(f"Excluded datasets: {sorted(EXCLUDED_DATASETS)}")
print(f"Training datasets: {DATASETS}")
for d, m in MODEL_MAP.items():
    print(f"  {d:10s} -> {m}")
print(f"Run tag: {RUN_TAG}")
print(f"Default Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}, Patience: {PATIENCE}")
print(f"Weight decay: {WEIGHT_DECAY}, Label smoothing: {LABEL_SMOOTHING}, Grad clip: {GRAD_CLIP}")
print("Results dir: _results_braindecode/")

---
## 2. 安装 Braindecode

如果已经在当前环境装过，会自动跳过。

In [ ]:
# Kaggle/offline-friendly dependency setup
import importlib
import sys
import subprocess
from pathlib import Path

WHEEL_DIR_CANDIDATES = [
    Path.cwd() / "wheels",
    Path("/kaggle/working/wheels"),
    Path("/kaggle/input/ml-project-braindecode-wheels"),
    Path("/kaggle/input/ml-project-braindecode-wheels/wheels"),
]
WHEEL_ZIP = Path("/kaggle/input/ml-project-braindecode-wheels/wheels.zip")
if WHEEL_ZIP.exists() and not Path("/kaggle/working/wheels").exists():
    import zipfile
    Path("/kaggle/working/wheels").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(WHEEL_ZIP, "r") as zf:
        zf.extractall("/kaggle/working/wheels")

OFFLINE_PACKAGES = [
    "mne",
    "mne-bids",
    "skorch",
    "wfdb",
    "linear-attention-transformer",
    "docstring-inheritance",
    "torchinfo",
    "rotary-embedding-torch",
    "braindecode",
]

def _local_wheel_dir():
    for p in WHEEL_DIR_CANDIDATES:
        if p.exists() and any(p.glob("*.whl")):
            return p
    return None

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        wheel_dir = _local_wheel_dir()
        if wheel_dir is not None:
            print(f"Installing {pip_name} from local wheels: {wheel_dir}")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--no-index", "--find-links", str(wheel_dir),
                *OFFLINE_PACKAGES,
            ])
        else:
            print("Available /kaggle/input dirs:", [str(p) for p in Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else [])
            print(f"Installing {pip_name} from PyPI...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return importlib.import_module(import_name)

torch = ensure_package("torch")
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("h5py")
ensure_package("sklearn", "scikit-learn")
braindecode = ensure_package("braindecode")

print(f"PyTorch: {torch.__version__}")
print(f"Braindecode: {braindecode.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name}")
    print(f"GPU capability: sm_{capability[0]}{capability[1]}")
    if REQUIRE_GPU and capability < (7, 0):
        raise RuntimeError(
            f"GPU {gpu_name} has compute capability sm_{capability[0]}{capability[1]}, "
            "but the Kaggle PyTorch build requires sm_70 or newer. Select a T4 GPU instead of P100."
        )
elif REQUIRE_GPU:
    raise RuntimeError("GPU is required for this Kaggle run. Enable Accelerator=GPU in Kaggle Settings before rerunning.")
else:
    print("GPU not detected; continuing because REQUIRE_GPU=False.")


---
## 3. 数据路径自动探测

与原 notebook 逻辑相同，但专门针对 Braindecode 实验。

In [ ]:
# ================================================================
# 路径探测（与原版一致，但结果写到 _results_braindecode）
# ================================================================
import json
from pathlib import Path

_DATASET_MARKERS = ("BCIC2A", "CHINESE", "MDD", "SEED", "SLEEP")


def _is_valid_data_root(p: Path) -> bool:
    if not p.is_dir():
        return False
    return any((p / name / "train.h5").exists() for name in _DATASET_MARKERS)


def _data_root_candidates():
    drive = Path("/content/drive/MyDrive")
    candidates = [
        Path("/kaggle/input/ml-project-eeg-data/ml_project_data"),
        Path("/kaggle/input/mlproject-data/ml_project_data"),
        Path.cwd() / "data" / "course project",
        Path.cwd().parent / "data" / "course project",
        Path.cwd().parent.parent / "data" / "course project",
        Path("/content/ML_project/data/course project"),
        Path("/content/course project"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        print("Kaggle input roots:", [p.name for p in kaggle_input.iterdir() if p.is_dir()])
        for train_file in kaggle_input.glob("**/BCIC2A/train.h5"):
            # train_file = <data_root>/BCIC2A/train.h5
            candidates.append(train_file.parent.parent)

    if drive.exists():
        candidates.extend([
            drive / "ML_project" / "data" / "course project",
            drive / "ML_project" / "course project",
            drive / "course project",
            drive / "新建文件夹" / "course project",
            drive / "新建文件夹" / "ML_project" / "data" / "course project",
        ])
        for child in drive.iterdir():
            if not child.is_dir() or child.name.startswith("."):
                continue
            candidates.append(child / "data" / "course project")
            candidates.append(child / "course project")
    seen, unique = set(), []
    for p in candidates:
        key = str(p.resolve()) if p.exists() else str(p)
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def resolve_data_root():
    override = DATA_ROOT_OVERRIDE
    if override not in (None, ""):
        p = Path(override)
        if _is_valid_data_root(p):
            return p
        print(f"DATA_ROOT_OVERRIDE set but invalid, falling back to auto-detect: {p}")

    env_root = os.environ.get("EEG_DATA_ROOT")
    if env_root:
        p = Path(env_root)
        if _is_valid_data_root(p):
            return p

    tried = []
    for p in _data_root_candidates():
        tried.append(str(p))
        if _is_valid_data_root(p):
            return p

    raise FileNotFoundError(
        "Cannot locate dataset root.\n"
        "Tried:\n  - " + "\n  - ".join(tried)
    )


DATA_ROOT = resolve_data_root()
_DRIVE_DATA_SOURCE = DATA_ROOT if "drive/MyDrive" in str(DATA_ROOT) else None

# 结果目录（独立于原 _results/）
# 优先用项目根下的 _results_braindecode/
RESULTS_DIR_BASE = Path("/kaggle/working/_results_braindecode") if Path("/kaggle/working").exists() else Path.cwd().parent / "_results_braindecode"
if not RESULTS_DIR_BASE.parent.exists():
    # 回退到 data 目录下
    RESULTS_DIR_BASE = Path.cwd() / "_results_braindecode"
RESULTS_DIR_BASE.mkdir(parents=True, exist_ok=True)
print(f"Results dir: {RESULTS_DIR_BASE}")

# 记下 Drive 路径用于回写
DRIVE_ROOT = None
if "drive/MyDrive" in str(DATA_ROOT):
    DRIVE_ROOT = DATA_ROOT
    print(f"Drive 数据根目录: {DRIVE_ROOT}")

print(f"Data root: {DATA_ROOT}")

# 复制到本地缓存加速
if USE_LOCAL_CACHE and (not Path("/kaggle/input").exists()) and Path("/content").exists():
    import shutil
    for DATA_NAME in DATASETS:
        src_base = DATA_ROOT / DATA_NAME
        dst_base = Path("/content/course project") / DATA_NAME
        dst_base.mkdir(parents=True, exist_ok=True)
        for fn in ["train.h5", "val.h5", "test_x_only.h5",
                   "dataset_info.json", "dataset_info_fixed.json"]:
            src = src_base / fn
            dst = dst_base / fn
            if src.exists() and not dst.exists():
                shutil.copy2(src, dst)
                print(f"  Copied {DATA_NAME}/{fn}")
    DATA_ROOT = Path("/content/course project")
    print(f"Using local cache at: {DATA_ROOT}")

# 反向探测 Drive 路径
if DRIVE_ROOT is None and _DRIVE_DATA_SOURCE is not None:
    DRIVE_ROOT = _DRIVE_DATA_SOURCE
elif DRIVE_ROOT is None and (not Path("/kaggle/input").exists()) and Path("/content/drive/MyDrive").exists():
    for candidate in _data_root_candidates():
        if "drive/MyDrive" in str(candidate) and _is_valid_data_root(candidate):
            DRIVE_ROOT = candidate
            print(f"Detected Drive path: {DRIVE_ROOT}")
            break

---
## 4. 导入 data_adapter（含内联后备）

优先从 `src/data_adapter.py` 导入；如果 Colab 路径找不到，则使用内联定义。

In [ ]:
# ================================================================
# 导入 data_adapter（优先从 src/，否则内联）
# ================================================================
try:
    # 从 src/ 导入（本地 / Drive 同步后）
    _src_candidates = [
        Path.cwd() / "src",
        Path.cwd().parent / "src",
        Path("/content/drive/MyDrive/ML_project/src"),
        Path("/content/ML_project/src"),
    ]
    for _p in _src_candidates:
        if _p.exists():
            sys.path.insert(0, str(_p))
    from data_adapter import (
        load_dataset, get_dataloaders, create_model, count_params,
        BraindecodeModelWrapper,
    )
    print("✓ Loaded data_adapter from src/data_adapter.py")
except ImportError as e:
    print(f"src/data_adapter.py not found ({e}), using inline fallback.")

    # ─── 内联后备定义 ────────────────────────────────────────
    import h5py
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    class H5Dataset(Dataset):
        def __init__(self, x, y=None):
            self.x = torch.tensor(x, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        def __len__(self):
            return len(self.x)
        def __getitem__(self, idx):
            if self.y is not None:
                return self.x[idx], self.y[idx]
            return self.x[idx]

    def load_dataset(data_root, dataset_name):
        base = Path(data_root) / dataset_name
        info_path = base / "dataset_info.json"
        if not info_path.exists():
            info_path = base / "dataset_info_fixed.json"
        with open(info_path) as f:
            info = json.load(f)
        categories = info["dataset"]["category_list"]
        sfreq = info["processing"]["target_sampling_rate"]
        def _load_h5(path):
            with h5py.File(str(path), "r") as f:
                keys = list(f.keys())
                X = f["X"][()].astype(np.float32)
                y = f["y"][()].astype(np.int64) if "y" in keys else None
            return X, y
        X_train, y_train = _load_h5(base / "train.h5")
        X_val, y_val = _load_h5(base / "val.h5")
        X_test, _ = _load_h5(base / "test_x_only.h5")
        n_channels, n_time = X_train.shape[1], X_train.shape[2]
        metadata = {
            "name": dataset_name, "categories": categories,
            "num_classes": len(categories),
            "n_channels": n_channels, "n_time": n_time,
            "sfreq": sfreq, "window_sec": n_time / sfreq,
            "n_train": len(X_train), "n_val": len(X_val), "n_test": len(X_test),
        }
        return X_train, y_train, X_val, y_val, X_test, metadata

    def get_dataloaders(X_train, y_train, X_val, y_val, X_test, bs=32):
        tr = DataLoader(H5Dataset(X_train, y_train), batch_size=bs, shuffle=True)
        vl = DataLoader(H5Dataset(X_val, y_val), batch_size=bs, shuffle=False)
        te = DataLoader(H5Dataset(X_test), batch_size=1, shuffle=False)
        return tr, vl, te

    class BraindecodeModelWrapper(nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model
        def forward(self, x):
            return self.model(x)

    def create_model(dataset_name, metadata):
        from braindecode.models import ShallowFBCSPNet, Deep4Net, USleep
        import inspect

        n = dataset_name.upper()
        c = metadata["n_channels"]
        k = metadata["num_classes"]
        s = metadata["sfreq"]
        t = metadata["n_time"]

        def _model_kwargs(model_class, kwargs):
            params = inspect.signature(model_class).parameters
            adapted = dict(kwargs)
            if "n_chans" in adapted and "n_chans" not in params and "in_chans" in params:
                adapted["in_chans"] = adapted.pop("n_chans")
            if "input_window_samples" in adapted and "input_window_samples" not in params and "n_times" in params:
                adapted["n_times"] = adapted.pop("input_window_samples")
            if "n_classes" in adapted and "n_classes" not in params and "n_outputs" in params:
                adapted["n_outputs"] = adapted.pop("n_classes")
            if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values()):
                adapted = {key: value for key, value in adapted.items() if key in params}
            return adapted

        model_map = {
            "SLEEP": (USleep, {
                "n_chans": c, "n_outputs": k, "n_classes": k,
                "sfreq": s, "input_window_seconds": t / s,
                "ensure_odd_conv_size": True,
            }),
            "BCIC2A": (ShallowFBCSPNet, {
                "n_chans": c, "n_outputs": k, "n_classes": k,
                "n_times": t, "input_window_samples": t,
            }),
            "MDD": (Deep4Net, {
                "n_chans": c, "n_outputs": k, "n_classes": k,
                "n_times": t, "input_window_samples": t,
            }),
            "SEED": (Deep4Net, {
                "n_chans": c, "n_outputs": k, "n_classes": k,
                "n_times": t, "input_window_samples": t,
            }),
            "CHINESE": (Deep4Net, {
                "n_chans": c, "n_outputs": k, "n_classes": k,
                "n_times": t, "input_window_samples": t,
            }),
        }
        if n not in model_map:
            raise ValueError(f"Unknown dataset {dataset_name!r}; choose from {sorted(model_map)}")
        cls, kw = model_map[n]
        return BraindecodeModelWrapper(cls(**_model_kwargs(cls, kw)))

    def count_params(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("✓ Inline fallback loaded.")

---
## 5. 训练与评估辅助函数

In [ ]:
# ================================================================
# 训练一个实验（数据集 × 模型），返回指标 + 保存结果
# ================================================================
import time
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datetime import datetime
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


set_global_seed(SEED)


def train_one_experiment(
    dataset_name,
    model_name,
    X_train, y_train,
    X_val,   y_val,
    X_test,
    metadata,
    exp_dir,
    epochs=50,
    batch_size=32,
    lr=1e-3,
    patience=10,
    weight_decay=1e-4,
    label_smoothing=0.05,
    grad_clip=1.0,
    scheduler_name="cosine",
    seed=42,
    model=None,
):
    """训练一个 Braindecode 模型并保存完整结果到 exp_dir。"""
    exp_dir = Path(exp_dir)
    exp_dir.mkdir(parents=True, exist_ok=True)
    num_classes = metadata["num_classes"]
    category_list = metadata["categories"]

    # 构建模型（允许外部传入覆盖）
    if model is None:
        model = create_model(dataset_name, metadata).to(device)
    else:
        model = model.to(device)
    n_params = count_params(model)
    print(f"    Model params: {n_params:,}")

    # DataLoaders
    train_loader, val_loader, test_loader = get_dataloaders(
        X_train, y_train, X_val, y_val, X_test, batch_size)

    # 损失 & 优化器
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = None
    if scheduler_name == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, epochs), eta_min=lr * 0.05)
    elif scheduler_name not in (None, "none"):
        raise ValueError(f"Unsupported scheduler: {scheduler_name}")

    # ── 训练循环 ──
    train_losses, val_losses, val_accs, lr_history = [], [], [], []
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0
    t0 = time.time()

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss_sum, train_num = 0.0, 0
        for data, label in train_loader:
            data, label = data.to(device), label.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, label)
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            train_loss_sum += loss.item() * label.size(0)
            train_num += label.size(0)
        if scheduler is not None:
            scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        lr_history.append(current_lr)
        epoch_train_loss = train_loss_sum / train_num
        train_losses.append(epoch_train_loss)

        # Val
        model.eval()
        val_loss_sum, val_correct, val_num = 0.0, 0, 0
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for val_data, val_label in val_loader:
                val_data, val_label = val_data.to(device), val_label.to(device)
                val_output = model(val_data)
                val_loss = criterion(val_output, val_label)
                val_loss_sum += val_loss.item() * val_label.size(0)
                val_num += val_label.size(0)
                val_pred = torch.argmax(val_output, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                all_val_preds.extend(val_pred.cpu().tolist())
                all_val_labels.extend(val_label.cpu().tolist())
        epoch_val_loss = val_loss_sum / val_num
        epoch_val_acc = val_correct / val_num
        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc)

        # Best checkpoint
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
                'val_acc': epoch_val_acc,
                'val_loss': epoch_val_loss,
            }, exp_dir / "best_model.pt")
        else:
            patience_counter += 1

        # Log
        if (epoch + 1) % 5 == 0 or epoch == epochs - 1 or epoch == 0:
            print(f"      Epoch [{epoch+1:02d}/{epochs}] | "
                  f"Train Loss: {epoch_train_loss:.4f} | "
                  f"Val Loss: {epoch_val_loss:.4f} | "
                  f"Val Acc: {epoch_val_acc:.4f} | "
                  f"LR: {current_lr:.2e} | "
                  f"Best: {best_val_acc:.4f} (ep {best_epoch})")

        # Early stopping
        if patience_counter >= patience:
            print(f"      Early stopping at epoch {epoch+1}")
            break

    elapsed = time.time() - t0
    final_val_acc = val_accs[-1]
    best_val_acc_epoch = best_epoch

    print(f"    ✔ Done | {elapsed:.1f}s | Best Val Acc: {best_val_acc:.4f} (ep {best_epoch})")

    # ── 加载最佳模型 ──
    checkpoint = torch.load(exp_dir / "best_model.pt", map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()


    best_val_loss_sum, best_val_correct, best_val_num = 0.0, 0, 0
    best_val_preds, best_val_labels = [], []
    with torch.no_grad():
        for val_data, val_label in val_loader:
            val_data, val_label = val_data.to(device), val_label.to(device)
            val_output = model(val_data)
            val_loss = criterion(val_output, val_label)
            best_val_loss_sum += val_loss.item() * val_label.size(0)
            best_val_num += val_label.size(0)
            val_pred = torch.argmax(val_output, dim=1)
            best_val_correct += (val_pred == val_label).sum().item()
            best_val_preds.extend(val_pred.cpu().tolist())
            best_val_labels.extend(val_label.cpu().tolist())
    best_checkpoint_val_loss = best_val_loss_sum / best_val_num
    best_checkpoint_val_acc = best_val_correct / best_val_num
    all_val_preds = best_val_preds
    all_val_labels = best_val_labels
    print(f"      Best checkpoint Val Loss: {best_checkpoint_val_loss:.4f} | "
          f"Val Acc: {best_checkpoint_val_acc:.4f}")

    # ── 保存 metrics.csv ──
    metrics_df = pd.DataFrame({
        "epoch": list(range(1, len(train_losses) + 1)),
        "train_loss": train_losses,
        "val_loss": val_losses,
        "val_accuracy": val_accs,
        "learning_rate": lr_history,
    })
    metrics_df.to_csv(exp_dir / "metrics.csv", index=False)

    # ── 保存 loss_curve.png ──
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses)+1), train_losses, 'o-', label="Train Loss", linewidth=2, markersize=4)
    plt.plot(range(1, len(val_losses)+1),   val_losses,   's-', label="Val Loss",   linewidth=2, markersize=4)
    plt.axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f"Best epoch ({best_epoch})")
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.title(f"{dataset_name} / {model_name} — Loss Curve (Braindecode)", fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(exp_dir / "loss_curve.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 验证集混淆矩阵 ──
    cm = confusion_matrix(all_val_labels, all_val_preds)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title(f"{dataset_name} / {model_name} — Confusion Matrix", fontsize=14)
    plt.colorbar(shrink=0.8)
    tick_marks = np.arange(len(category_list))
    plt.xticks(tick_marks, category_list, rotation=45, ha='right')
    plt.yticks(tick_marks, category_list)
    plt.xlabel("Predicted", fontsize=12)
    plt.ylabel("True", fontsize=12)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.savefig(str(exp_dir / "val_confusion_matrix.png"), dpi=150, bbox_inches='tight')
    plt.close()

    # ── 分类报告 ──
    report_dict = classification_report(
        all_val_labels, all_val_preds,
        target_names=category_list, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(exp_dir / "classification_report.csv", float_format='%.4f')

    # ── Failure cases ──
    all_val_preds_arr = np.array(all_val_preds)
    all_val_labels_arr = np.array(all_val_labels)
    failure_mask = all_val_preds_arr != all_val_labels_arr
    if failure_mask.sum() > 0:
        failure_indices = np.where(failure_mask)[0]
        failure_df = pd.DataFrame({
            "sample_index": failure_indices,
            "true_label": all_val_labels_arr[failure_indices],
            "true_category": [category_list[l] for l in all_val_labels_arr[failure_indices]],
            "predicted_label": all_val_preds_arr[failure_indices],
            "predicted_category": [category_list[l] for l in all_val_preds_arr[failure_indices]],
        })
        failure_df.to_csv(exp_dir / "failure_cases.csv", index=False)
        print(f"      Failure cases: {len(failure_df)} samples")

    # ── Test 预测 ──
    all_test_preds = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data = test_data.to(device)
            test_output = model(test_data)
            test_pred = torch.argmax(test_output, dim=1)
            all_test_preds.extend(test_pred.cpu().tolist())

    with open(exp_dir / "predictions.txt", "w") as f:
        for label in all_test_preds:
            f.write(f"{int(label)}\n")
    print(f"      Saved {len(all_test_preds)} test predictions")

    # ── 返回汇总 ──
    return {
        "dataset": dataset_name,
        "model": model_name,
        "best_val_acc": round(best_val_acc, 4),
        "final_val_acc": round(final_val_acc, 4),
        "best_epoch": best_val_acc_epoch,
        "train_loss_final": round(train_losses[-1], 4),
        "epochs_run": len(train_losses),
        "time_sec": round(elapsed, 1),
        "num_params": n_params,
        "weight_decay": weight_decay,
        "label_smoothing": label_smoothing,
        "grad_clip": grad_clip,
        "scheduler": scheduler_name,
        "best_checkpoint_val_loss": round(best_checkpoint_val_loss, 4),
        "best_checkpoint_val_acc": round(best_checkpoint_val_acc, 4),
    }

---
## 6. 主实验循环

遍历所有数据集，用对应的 Braindecode 模型训练。

In [ ]:
# ================================================================
# Kaggle preflight: validate data and model construction before long training
# ================================================================
def run_preflight_checks():
    print("\n=== Kaggle preflight checks ===")
    checked = []
    for data_name in DATASETS:
        if data_name in EXCLUDED_DATASETS:
            print(f"[PREFLIGHT SKIP] {data_name}: excluded")
            continue
        model_name = MODEL_MAP[data_name]
        X_train, y_train, X_val, y_val, X_test, metadata = load_dataset(DATA_ROOT, data_name)
        required = ["n_train", "n_val", "n_test", "n_channels", "n_time", "num_classes", "sfreq", "categories"]
        missing = [key for key in required if key not in metadata]
        if missing:
            raise KeyError(f"{data_name} metadata missing fields: {missing}")
        if metadata["n_train"] != len(X_train) or metadata["n_val"] != len(X_val) or metadata["n_test"] != len(X_test):
            raise ValueError(f"{data_name} metadata sample counts do not match loaded arrays")
        model = create_model(data_name, metadata).to(device)
        model.eval()
        sample = torch.as_tensor(X_train[:1], dtype=torch.float32, device=device)
        with torch.no_grad():
            output = model(sample)
        if output.shape[0] != 1 or output.shape[-1] != metadata["num_classes"]:
            raise ValueError(
                f"{data_name}/{model_name} output shape {tuple(output.shape)} does not match "
                f"num_classes={metadata['num_classes']}"
            )
        checked.append({
            "dataset": data_name,
            "model": model_name,
            "shape": tuple(X_train.shape),
            "output_shape": tuple(output.shape),
        })
        print(f"[PREFLIGHT OK] {data_name}/{model_name}: X={tuple(X_train.shape)}, output={tuple(output.shape)}")
        del model, sample, output
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print("=== Kaggle preflight complete ===")
    return checked

if PREFLIGHT_ONLY:
    preflight_results = run_preflight_checks()
    print("PREFLIGHT_ONLY=True; downstream training cells will be skipped to protect Kaggle quota.")


In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 15: formal-training-only code skipped.")
else:
    # ================================================================
    # Main loop: all leaderboard datasets and their selected SOTA models
    # ================================================================

    results_summary = []
    if RUN_TAG == "formal":
        experiments_csv_path = RESULTS_DIR_BASE / "experiments_summary_formal.csv"
        summary_json_path = RESULTS_DIR_BASE / "summary_formal.json"
    else:
        experiments_csv_path = RESULTS_DIR_BASE / f"experiments_summary_{RUN_TAG}.csv"
        summary_json_path = RESULTS_DIR_BASE / f"summary_{RUN_TAG}.json"

    for DATA_NAME in DATASETS:
        if DATA_NAME in EXCLUDED_DATASETS:
            print(f"[SKIP] {DATA_NAME}: excluded from leaderboard")
            continue

        MODEL_NAME = MODEL_MAP[DATA_NAME]
        run_cfg = {
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "grad_clip": GRAD_CLIP,
            "scheduler_name": "cosine",
            "seed": SEED,
        }
        run_cfg.update(DATASET_TRAINING_OVERRIDES.get(DATA_NAME, {}))

        print(f"\n{'='*60}")
        print(f"Dataset: {DATA_NAME}  -> Model: {MODEL_NAME}")
        print(f"{'='*60}")
        print(f"  Training config: {run_cfg}")

        # Only formal runs can block formal reruns; old smoke1ep dirs are ignored.
        exp_parent = RESULTS_DIR_BASE / DATA_NAME / MODEL_NAME
        existing_runs = sorted([
            d.name for d in exp_parent.iterdir()
            if d.is_dir() and RUN_TAG in d.name
        ]) if exp_parent.exists() else []
        if SKIP_EXISTING and existing_runs:
            print(f"  [SKIP] {DATA_NAME}/{MODEL_NAME} already has {len(existing_runs)} {RUN_TAG} run(s)")
            results_summary.append({
                "dataset": DATA_NAME, "model": MODEL_NAME,
                "val_acc": None, "best_val_acc": None, "status": "skipped"
            })
            continue

        X_train, y_train, X_val, y_val, X_test, metadata = load_dataset(DATA_ROOT, DATA_NAME)

        print(f"  Shape: (N, C={metadata['n_channels']}, T={metadata['n_time']})")
        print(f"  Classes: {metadata['categories']}")
        print(f"  sfreq: {metadata['sfreq']} Hz, window: {metadata['window_sec']}s")
        print(f"  Train: {metadata['n_train']}, Val: {metadata['n_val']}, Test: {metadata['n_test']}")

        exp_parent.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        exp_name = f"001_{timestamp}_{RUN_TAG}_bs{run_cfg['batch_size']}_lr{run_cfg['lr']}"
        exp_dir = exp_parent / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)
        print(f"  Experiment dir: {exp_dir}")

        config = {
            "dataset": DATA_NAME,
            "model": MODEL_NAME,
            "run_tag": RUN_TAG,
            "batch_size": run_cfg["batch_size"],
            "epochs": run_cfg["epochs"],
            "learning_rate": run_cfg["lr"],
            "optimizer": "AdamW",
            "weight_decay": run_cfg["weight_decay"],
            "label_smoothing": run_cfg["label_smoothing"],
            "grad_clip": run_cfg["grad_clip"],
            "scheduler": run_cfg["scheduler_name"],
            "loss_fn": "CrossEntropyLoss",
            "seed": run_cfg["seed"],
            "num_classes": metadata["num_classes"],
            "channels": metadata["n_channels"],
            "time_points": metadata["n_time"],
            "sfreq": metadata["sfreq"],
            "categories": metadata["categories"],
            "experiment_time": timestamp,
            "braindecode_model": MODEL_NAME,
        }
        with open(exp_dir / "config.json", "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)

        result = train_one_experiment(
            DATA_NAME, MODEL_NAME,
            X_train, y_train,
            X_val,   y_val,
            X_test,
            metadata,
            exp_dir,
            **run_cfg,
        )
        result["status"] = "done"
        result["run_tag"] = RUN_TAG
        result["experiment_dir"] = str(exp_dir)
        result["experiment_name"] = exp_name
        results_summary.append(result)

        entry_df = pd.DataFrame([result])
        if experiments_csv_path.exists():
            existing_df = pd.read_csv(experiments_csv_path)
            combined = pd.concat([existing_df, entry_df], ignore_index=True)
            combined.to_csv(experiments_csv_path, index=False)
        else:
            entry_df.to_csv(experiments_csv_path, index=False)

        with open(summary_json_path, "w", encoding="utf-8") as f:
            json.dump(results_summary, f, indent=2, ensure_ascii=False)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print("All Braindecode formal experiments finished.")
    print(f"CSV: {experiments_csv_path}")
    print(f"JSON: {summary_json_path}")
    print(f"{'='*60}")

---
## 7. 结果汇总表

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 17: formal-training-only code skipped.")
else:
    # ================================================================
    # 结果汇总表
    # ================================================================

    print(f"{'数据集':<12} {'模型':<20} {'Best Acc':<12} {'Val Acc':<12} {'Params':<10} {'Time':<8}")
    print("-" * 74)

    for r in results_summary:
        if r.get("status") == "skipped":
            print(f"{r['dataset']:<12} {r['model']:<20} {'[skipped]':<12}")
            continue
        best = f"{r['best_val_acc']:.4f}"
        val  = f"{r['final_val_acc']:.4f}"
        params = f"{r.get('num_params', 0):,}"
        t = f"{r['time_sec']:.0f}s"
        print(f"{r['dataset']:<12} {r['model']:<20} {best:<12} {val:<12} {params:<10} {t:<8}")

    print("-" * 74)

    # 找出最佳
    valid = [r for r in results_summary if r.get("status") != "skipped" and r.get("best_val_acc")]
    if valid:
        best = max(valid, key=lambda r: r["best_val_acc"])
        print(f"\n🏆 最佳: {best['dataset']} / {best['model']} = {best['best_val_acc']:.4f}")

    # 保存内存中汇总到 JSON
    with open(summary_json_path, "w") as f:
        json.dump(results_summary, f, indent=2, ensure_ascii=False)
    print(f"\n汇总已保存到: {summary_json_path}")

---
## 8. 与原始 EEGNet 基线对比 🔥

自动加载 `_results/summary.json` 中的旧结果进行并排对比。

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 19: formal-training-only code skipped.")
else:
    # ================================================================
    # 对比：Braindecode vs 原始基线
    # ================================================================

    # 读取旧结果
    old_results_path = DATA_ROOT.parent / "_results" / "summary.json"
    if not old_results_path.exists():
        # 尝试项目根目录
        old_results_path = Path.cwd().parent / "_results" / "summary.json"

    if old_results_path.exists():
        with open(old_results_path) as f:
            old_results = json.load(f)

        # 取每个数据集的最佳旧结果
        old_best = {}
        for r in old_results:
            ds = r["dataset"]
            if ds in EXCLUDED_DATASETS:
                continue
            ba = r.get("best_val_acc")
            if ba is None:
                continue
            if ds not in old_best or ba > old_best[ds]["best_val_acc"]:
                old_best[ds] = r

        # 取每个数据集的新结果
        new_best = {}
        for r in results_summary:
            ds = r["dataset"]
            if ds in EXCLUDED_DATASETS:
                continue
            ba = r.get("best_val_acc")
            if ba is None:
                continue
            if ds not in new_best or ba > new_best[ds]["best_val_acc"]:
                new_best[ds] = r

        print(f"{'数据集':<12} {'旧最佳':<20} {'旧 Acc':<12} {'新最佳':<20} {'新 Acc':<12} {'提升':<12}")
        print("-" * 88)

        improvements = []
        for ds in DATASETS:
            if ds in EXCLUDED_DATASETS:
                continue
            old_r = old_best.get(ds, {})
            new_r = new_best.get(ds, {})
            old_model = old_r.get("model", "?")
            new_model = new_r.get("model", "?")
            old_acc = old_r.get("best_val_acc", "?")
            new_acc = new_r.get("best_val_acc", "?")

            if isinstance(old_acc, (int, float)) and isinstance(new_acc, (int, float)):
                delta = new_acc - old_acc
                delta_str = f"{delta:+.2%}" if delta < 0.1 else f"{delta:+.2%} 🔥"
                improvements.append(delta)
            else:
                delta_str = "?"

            old_label = f"{old_model}" if old_model else "(no data)"
            new_label = f"{new_model}" if new_model else "(no data)"
            old_acc_str = f"{old_acc:.2%}" if isinstance(old_acc, (int, float)) else "N/A"
            new_acc_str = f"{new_acc:.2%}" if isinstance(new_acc, (int, float)) else "N/A"

            print(f"{ds:<12} {old_label:<20} {old_acc_str:<12} {new_label:<20} {new_acc_str:<12} {delta_str:<12}")

        print("-" * 88)
        if improvements:
            avg_improvement = sum(improvements) / len(improvements)
            print(f"\n📊 平均提升: {avg_improvement:+.2%}")
            print(f"   数据集数: {len(improvements)}")
    else:
        print(f"未找到旧结果文件: {old_results_path}")
        print("请先运行原 notebook 生成 _results/summary.json")

---
## 9. 最佳模型对比可视化

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 21: formal-training-only code skipped.")
else:
    # ================================================================
    # 柱状图对比
    # ================================================================
    import matplotlib.pyplot as plt

    if old_results_path.exists() and 'old_best' in dir() and improvements:
        datasets_labels = []
        old_accs = []
        new_accs = []
        for ds in DATASETS:
            if ds in EXCLUDED_DATASETS:
                continue
            old_r = old_best.get(ds)
            new_r = new_best.get(ds)
            if old_r and new_r:
                oa = old_r.get("best_val_acc")
                na = new_r.get("best_val_acc")
                if isinstance(oa, (int, float)) and isinstance(na, (int, float)):
                    datasets_labels.append(ds)
                    old_accs.append(oa * 100)
                    new_accs.append(na * 100)

        if datasets_labels:
            x = np.arange(len(datasets_labels))
            width = 0.35

            fig, ax = plt.subplots(figsize=(12, 6))
            bars1 = ax.bar(x - width/2, old_accs, width, label='EEGNet / SimpleLinear / MLP', color='#6baed6')
            bars2 = ax.bar(x + width/2, new_accs, width, label='Braindecode SOTA', color='#fd8d3c')

            ax.set_xlabel('Dataset')
            ax.set_ylabel('Best Validation Accuracy (%)')
            ax.set_title('Braindecode SOTA vs Original Baselines')
            ax.set_xticks(x)
            ax.set_xticklabels(datasets_labels)
            ax.legend()
            ax.axhline(y=100/len(category_list) if 'category_list' in dir() else 50,
                       color='gray', linestyle=':', alpha=0.5, label='Chance')
            ax.grid(axis='y', alpha=0.3)

            # 在柱上标值
            for bar in bars1:
                ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                        f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
            for bar in bars2:
                ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                        f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

            plt.tight_layout()
            plt.savefig(str(RESULTS_DIR_BASE / f"comparison_barplot_{RUN_TAG}.png"), dpi=150, bbox_inches='tight')
            plt.show()
            print(f"对比图已保存: {RESULTS_DIR_BASE / f'comparison_barplot_{RUN_TAG}.png'}")

---
## 10. 消融实验

选择最佳表现的数据集 × 模型组合，测试不同超参数/架构变体的影响。

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 23: formal-training-only code skipped.")
else:
    # ================================================================
    # 消融实验配置
    # ================================================================

    # 挑选做消融的数据集
    ABLATION_DATASET = "SLEEP"
    ABLATION_BASE_NAME = MODEL_MAP[ABLATION_DATASET]  # USleep

    # 消融配置：修改 Braindecode 模型的关键参数
    # 注意：不同模型的参数不同，这里以 USleep 为例
    ablation_configs = [
        {"name": "Baseline", "model_kwargs": {}},
        {"name": "No weight decay", "model_kwargs": {}, "optim_kwargs": {"weight_decay": 0}},
        {"name": "LR=1e-4", "model_kwargs": {}, "optim_kwargs": {"lr": 1e-4}},
    ]

    # 对 BCIC2A 额外做消融
    if ABLATION_DATASET == "BCIC2A":
        ablation_configs.append({"name": "Deep4Net instead", "model_kwargs": {}, "model_override": "Deep4Net"})

    print(f"消融实验: {ABLATION_BASE_NAME} @ {ABLATION_DATASET}")
    for cfg in ablation_configs:
        print(f"  {cfg['name']}")

    # ── 加载数据 ──
    X_train, y_train, X_val, y_val, X_test, meta = load_dataset(DATA_ROOT, ABLATION_DATASET)

    print(f"\n{'配置':<22} {'Val Acc':<12} {'Best Acc':<12} {'Params':<10}")
    print("-" * 56)

    ablation_results = []

    for cfg in ablation_configs:
        name = cfg["name"]
        model_override = cfg.get("model_override")
        optim_kw = cfg.get("optim_kwargs", {})

        # 构建模型
        prebuilt_model = None
        if model_override:
            # 替换模型类型
            from braindecode.models import Deep4Net, ShallowFBCSPNet, USleep
            model_cls = {
                "Deep4Net": Deep4Net,
                "ShallowFBCSPNet": ShallowFBCSPNet,
                "USleep": USleep,
            }[model_override]
            if model_override == "USleep":
                raw_model = model_cls(
                    n_chans=meta["n_channels"],
                    n_classes=meta["num_classes"],
                    sfreq=meta["sfreq"],
                    input_window_seconds=meta["n_time"] / meta["sfreq"],
                )
            else:
                raw_model = model_cls(
                    n_chans=meta["n_channels"],
                    n_classes=meta["num_classes"],
                    input_window_samples=meta["n_time"],
                )
            prebuilt_model = BraindecodeModelWrapper(raw_model) if model_override else None

        # 训练
        result = train_one_experiment(
            ABLATION_DATASET,
            f"{ABLATION_BASE_NAME}_{name}".replace(" ", "_"),
            X_train, y_train,
            X_val,   y_val,
            X_test,
            meta,
            exp_dir=RESULTS_DIR_BASE / "ablation" / f"{ABLATION_DATASET}_{name.replace(' ', '_')}",
            epochs=EPOCHS // 2,  # 消融跑少一些 epoch
            batch_size=BATCH_SIZE,
            lr=optim_kw.get("lr", LR),
            patience=PATIENCE // 2,
            model=prebuilt_model,
        )

        print(f"{name:<22} {result['final_val_acc']:<12.4f} {result['best_val_acc']:<12.4f} {result['num_params']:<10,}")
        ablation_results.append({
            "name": name,
            "model_kwargs": cfg.get("model_kwargs", {}),
            "optim_kwargs": cfg.get("optim_kwargs", {}),
            "final_val_acc": result["final_val_acc"],
            "best_val_acc": result["best_val_acc"],
            "num_params": result["num_params"],
        })

    # 保存消融结果
    ablation_path = RESULTS_DIR_BASE / f"ablation_{ABLATION_DATASET}_{ABLATION_BASE_NAME}.json"
    with open(ablation_path, "w") as f:
        json.dump({
            "dataset": ABLATION_DATASET,
            "base_model": ABLATION_BASE_NAME,
            "configs": ablation_results,
        }, f, indent=2, ensure_ascii=False)
    print(f"\n消融结果已保存: {ablation_path}")
    print("消融实验完成！")

---
## 11. 导出统一预测文件

将各数据集的预测结果整理到统一目录，格式与原版一致（每行一个整数标签）。

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 25: formal-training-only code skipped.")
else:
    # ================================================================
    # 导出预测结果到统一目录
    # ================================================================

    pred_dir = RESULTS_DIR_BASE / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)

    for DATA_NAME in DATASETS:
        if DATA_NAME in EXCLUDED_DATASETS:
            print(f"  [SKIP] {DATA_NAME}: excluded from leaderboard")
            continue
        MODEL_NAME = MODEL_MAP[DATA_NAME]

        # 找最新的实验目录
        exp_parent = RESULTS_DIR_BASE / DATA_NAME / MODEL_NAME
        if not exp_parent.exists():
            print(f"  [SKIP] {DATA_NAME}: no experiment directory")
            continue

        runs = sorted([d for d in exp_parent.iterdir() if d.is_dir() and RUN_TAG in d.name], reverse=True)
        if not runs:
            print(f"  [SKIP] {DATA_NAME}: no runs found")
            continue

        latest = runs[0]
        pred_file = latest / "predictions.txt"
        if not pred_file.exists():
            print(f"  [SKIP] {DATA_NAME}: predictions.txt not found in {latest.name}")
            continue

        # 复制到统一目录
        out_name = f"{DATA_NAME}_{MODEL_NAME}.txt"
        out_path = pred_dir / out_name
        import shutil
        shutil.copy2(pred_file, out_path)

        # 也复制到数据集目录下（兼容旧版读取位置）
        dst = DATA_ROOT / DATA_NAME / out_name
        try:
            if os.access(DATA_ROOT / DATA_NAME, os.W_OK):
                shutil.copy2(pred_file, dst)
        except Exception as exc:
            print(f"  Skip writing to read-only data root: {exc}")

        print(f"  ✓ {out_name}: {out_path}")

    print(f"\n所有预测文件已导出到: {pred_dir}")

---
## 12. 结果回写到 Google Drive

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 27: formal-training-only code skipped.")
else:
    # ================================================================
    # 回写结果到 Google Drive
    # ================================================================

    if SYNC_TO_DRIVE and DRIVE_ROOT is not None:
        import shutil

        print(f"\n正在回写结果到 Google Drive ...")
        print(f"  目标: {DRIVE_ROOT}")

        # 同步整个 _results_braindecode/ 目录树
        dst_root = DRIVE_ROOT / "_results_braindecode"
        dst_root.mkdir(parents=True, exist_ok=True)

        for item in RESULTS_DIR_BASE.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(RESULTS_DIR_BASE)
                dst_file = dst_root / rel_path
                dst_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, dst_file)

        # 统计
        txt_files = list(dst_root.rglob("*.txt"))
        csv_files = list(dst_root.rglob("*.csv"))
        pt_files = list(dst_root.rglob("*.pt"))
        png_files = list(dst_root.rglob("*.png"))
        json_files = list(dst_root.rglob("*.json"))
        print(f"  共: {len(txt_files)} 预测, {len(csv_files)} CSV, "
              f"{len(pt_files)} 模型, {len(png_files)} 图片, {len(json_files)} 配置")

        # 同步 summary CSV 到 Drive 根目录
        if experiments_csv_path.exists():
            shutil.copy2(experiments_csv_path, DRIVE_ROOT / f"experiments_summary_braindecode_{RUN_TAG}.csv")
            print(f"  汇总表已回写: experiments_summary_braindecode.csv")

        print(f"\n✅ 所有结果已回写到: {dst_root}")
    else:
        print("\n未启用 Drive 回写或未检测到 Drive 路径，跳过。")
        print(f"  SYNC_TO_DRIVE = {SYNC_TO_DRIVE}")
        print(f"  DRIVE_ROOT = {DRIVE_ROOT}")

    print("\n完成！")

In [ ]:
# PREFLIGHT_SKIP_WRAPPED
if PREFLIGHT_ONLY:
    print("[PREFLIGHT SKIP] Cell 28: formal-training-only code skipped.")
else:
    from pathlib import Path
    import os, sys

    print("cwd =", Path.cwd())
    print("notebook likely project root candidates:")
    for p in [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/ML_project"),
        Path("/content/drive/MyDrive/ML_project"),
    ]:
        print(p, "exists=", p.exists(), "src=", (p / "src").exists(), "notebooks=", (p / "notebooks").exists())

    import data_adapter
    print("data_adapter loaded from =", data_adapter.__file__)
    print("python =", sys.executable)